
# Card &middot; Descriptors and regressors

| | |
|---|---|
| **time** | ~45 minutes |
| **GPU** | not needed |
| **typical gain** | large &mdash; this is where most of the easy improvement is |
| **needs** | nothing; runs standalone |

How you *describe* a molecule to a model matters more than which model you
pick. This card is about that choice.

A warning about how to spend the time: it is tempting to run every descriptor
against every regressor and take the winner. Resist. That is 20 numbers and no
understanding, and you will not be able to explain any of it on slide 2. Pick
combinations you have a *reason* to expect will work, and write the reason
down first.

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid")

train = common.load_train()
test  = common.load_test()
fold, split_meta = common.load_split(train)
print("split in use:", split_meta.get("method"))

---
## 1. Three ways to describe a molecule

**RDKit descriptors** (~210 numbers) &mdash; interpretable physical quantities:
molecular weight, cLogP, topological polar surface area, rotatable bonds,
ring counts. These encode exactly the chemistry that drives ADMET, which is
why a boring model on these is such a strong baseline.

**Morgan fingerprints** (2048 bits) &mdash; a bit is set if a particular circular
substructure is present. Captures *what groups are there*, not bulk
properties. Good at recognising series, bad at extrapolating.

**Mordred** (~1600 descriptors) &mdash; a much bigger physicochemical set. Slow to
compute, so it is precomputed for you.

In [ ]:
X_rdkit_tr = common.rdkit_descriptors(train[common.SMILES_COL])
X_rdkit_te = common.rdkit_descriptors(test[common.SMILES_COL])

fp_tr = common.fingerprints_to_array(common.morgan_fingerprints(train[common.SMILES_COL]))
fp_te = common.fingerprints_to_array(common.morgan_fingerprints(test[common.SMILES_COL]))
X_morgan_tr = pd.DataFrame(fp_tr, columns=[f"bit_{i}" for i in range(fp_tr.shape[1])])
X_morgan_te = pd.DataFrame(fp_te, columns=X_morgan_tr.columns)

print("rdkit :", X_rdkit_tr.shape)
print("morgan:", X_morgan_tr.shape)

In [ ]:
# Mordred: precomputed, because computing it live costs ~10 minutes.
try:
    mord = common.load_artifact("mordred_descriptors.parquet")
    X_mordred_tr = mord.set_index(common.ID_COL).reindex(train[common.ID_COL]).reset_index(drop=True)
    X_mordred_te = mord.set_index(common.ID_COL).reindex(test[common.ID_COL]).reset_index(drop=True)
    print("mordred:", X_mordred_tr.shape)
except Exception as exc:
    X_mordred_tr = X_mordred_te = None
    print("mordred unavailable:", exc)

### Combining descriptor sets

Concatenating RDKit descriptors with fingerprints often beats either alone:
the descriptors carry bulk physics, the bits carry substructure.

In [ ]:
FEATURES = {
    "rdkit":        (X_rdkit_tr, X_rdkit_te),
    "morgan":       (X_morgan_tr, X_morgan_te),
    "rdkit+morgan": (pd.concat([X_rdkit_tr, X_morgan_tr], axis=1),
                     pd.concat([X_rdkit_te, X_morgan_te], axis=1)),
}
if X_mordred_tr is not None:
    FEATURES["mordred"] = (X_mordred_tr, X_mordred_te)
list(FEATURES)

---
## 2. Regressors

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

def make_model(kind):
    if kind == "ridge":
        return RidgeCV(alphas=np.logspace(-2, 4, 20))
    if kind == "rf":
        return RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0)
    if kind == "lgbm":
        return LGBMRegressor(n_estimators=500, learning_rate=0.05,
                             num_leaves=31, verbose=-1, n_jobs=-1)
    raise ValueError(kind)


def run(feature_name, model_kind, endpoints=None, verbose=True):
    """Fit on the train fold, score on the val fold. Returns (metrics, preds)."""
    endpoints = endpoints or common.ENDPOINTS
    Xtr_all, _ = FEATURES[feature_name]
    tr = (fold == "train").to_numpy(); va = (fold == "val").to_numpy()
    A, B = common.clean_features(Xtr_all[tr], Xtr_all[va])
    out = pd.DataFrame({common.ID_COL: train.loc[va, common.ID_COL].to_numpy()})
    for e in endpoints:
        y = train.loc[tr, e]; ok = y.notna().to_numpy()
        if ok.sum() < 50:
            out[e] = np.nan; continue
        m = make_model(model_kind)
        m.fit(A[ok], y[ok])
        out[e] = m.predict(B)
    ev = common.evaluate(train.loc[va].reset_index(drop=True), out)
    if verbose:
        print(f"{feature_name:14s} {model_kind:6s} MA-RAE = {ev['RAE'].mean():.3f}")
    return ev, out

### &#9654;&#65039; Predict first

**For LogD specifically: will RDKit descriptors or Morgan fingerprints win? For Log_Caco_ER (efflux)?**

*LogD is additive and driven by bulk properties. Efflux depends on whether P-gp recognises a specific 3D arrangement. Which representation suits which?*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

In [ ]:
scores = {}
for feat in ["rdkit", "morgan", "rdkit+morgan"]:
    ev, _ = run(feat, "lgbm")
    scores[feat] = ev["RAE"]

pd.DataFrame(scores).round(3)

Read that table **per endpoint**, not just by the average. Different
endpoints will prefer different representations, and noticing that is worth
more than the overall number.

Now vary the model with the representation held fixed.

In [ ]:
for kind in ["ridge", "rf", "lgbm"]:
    run("rdkit", kind)

Compare the spread you just saw from changing the *model* with the
spread from changing the *features*. In the real challenge, feature and data
engineering consistently beat model choice. Does that hold here?

---
## 3. TabPFN &mdash; a model that does not train

TabPFN is a transformer pre-trained on millions of synthetic tabular problems.
You hand it your training data at inference time and it predicts, with no
fitting step. It was popular in the real challenge, though it did not reach
the very top.

Two constraints worth knowing: it has limits on sample and feature count, so
it wants a compact feature set, and it much prefers a GPU.

In [ ]:
%pip -q install tabpfn
from tabpfn import TabPFNRegressor

# keep the feature set small -- TabPFN is happiest with tens, not thousands
from sklearn.feature_selection import SelectKBest, f_regression

def run_tabpfn(endpoint, k_features=32, max_n=3000):
    tr = (fold == "train").to_numpy(); va = (fold == "val").to_numpy()
    A, B = common.clean_features(X_rdkit_tr[tr], X_rdkit_tr[va])
    y = train.loc[tr, endpoint]; ok = y.notna().to_numpy()
    A, y = A[ok], y[ok]
    if len(A) > max_n:
        idx = np.random.default_rng(0).choice(len(A), max_n, replace=False)
        A, y = A.iloc[idx], y.iloc[idx]
    sel = SelectKBest(f_regression, k=min(k_features, A.shape[1])).fit(A, y)
    m = TabPFNRegressor()
    m.fit(sel.transform(A), y)
    p = m.predict(sel.transform(B))
    truth = train.loc[va].reset_index(drop=True)
    out = pd.DataFrame({common.ID_COL: truth[common.ID_COL], endpoint: p})
    return common.evaluate(truth, out, [endpoint])

run_tabpfn("LogD").round(3)

---
## 4. Hyperparameter optimisation

Here is the honest version of this section.

In the real challenge, several top-20 finishers ran extensive hyperparameter
searches and **most reported minimal gain**. The organisers' summary was that
additional data, data engineering and feature augmentation mattered more.

So rather than give you an HPO notebook, here is one cell. Run it, note how
long it took and what it bought, and compare that against what `card_external`
buys you in the same wall-clock time. That comparison *is* the lesson.

In [ ]:
import time
from sklearn.model_selection import RandomizedSearchCV

t0 = time.time()
tr = (fold == "train").to_numpy()
A = common.clean_features(X_rdkit_tr[tr])
y = train.loc[tr, "LogD"]; ok = y.notna().to_numpy()

search = RandomizedSearchCV(
    LGBMRegressor(verbose=-1, n_jobs=-1),
    {"n_estimators": [200, 400, 800, 1500],
     "learning_rate": [0.01, 0.03, 0.05, 0.1],
     "num_leaves": [15, 31, 63, 127],
     "min_child_samples": [5, 20, 50],
     "subsample": [0.7, 0.85, 1.0]},
    n_iter=25, cv=3, scoring="neg_mean_absolute_error", random_state=0, n_jobs=-1)
search.fit(A[ok], y[ok])

base = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31,
                     verbose=-1, n_jobs=-1)
from sklearn.model_selection import cross_val_score
base_mae = -cross_val_score(base, A[ok], y[ok], cv=3,
                            scoring="neg_mean_absolute_error").mean()

print(f"default settings : MAE {base_mae:.4f}")
print(f"after 25 configs : MAE {-search.best_score_:.4f}")
print(f"improvement      : {100 * (base_mae + search.best_score_) / base_mae:.1f}%")
print(f"time spent       : {time.time() - t0:.0f} s")

Write that improvement percentage on slide 2, whichever way it
came out. If it is large, you have found something the challenge participants
did not, and that is worth saying.

---
## Save your work

Give it a name you will recognise at 4pm. `card_ensembles` can combine this
with anything else you have made today.

In [ ]:
BEST_FEATURES = "rdkit+morgan"   # <-- your pick
BEST_MODEL    = "lgbm"           # <-- your pick

Xtr, Xte = FEATURES[BEST_FEATURES]
A, B = common.clean_features(Xtr, Xte)

pred = common.blank_predictions(test)
for e in common.ENDPOINTS:
    y = train[e]; ok = y.notna().to_numpy()
    m = make_model(BEST_MODEL)
    m.fit(A[ok], y[ok])
    pred[e] = m.predict(B)

common.save_predictions(pred, f"{BEST_MODEL}-{BEST_FEATURES}",
                        note=f"{BEST_MODEL} on {BEST_FEATURES}")